# Nature Methods Manuscript Figure Assembly

This notebook generates all figures and tables for the Nature Methods manuscript.

Phase 11: Reporting and manuscript figure assembly

In [ ]:
import json
import sys
from pathlib import Path

# Add src to path
src_path = Path.cwd().parent.parent / 'src'
if src_path not in sys.path:
    sys.path.insert(0, str(src_path))

from markovianity_diagnostic.reporting import (
    FIGURE_SPECS,
    generate_all_figures,
    generate_all_tables,
    create_manuscript_manifest,
)

# Set up paths
project_root = Path.cwd().parent.parent
print(f'Project root: {project_root}')

## Figure Specifications

Overview of all figures to be included in the manuscript:

In [ ]:
# Display figure specs
for fig_key, fig_spec in FIGURE_SPECS.items():
    print(f"\n{fig_spec['name']}")
    print(f"  Type: {fig_spec['fig_type']}")
    print(f"  Output: {fig_spec['output_path']}")
    print(f"  Sources: {len(fig_spec['source_data_paths'])} files")

## Generate All Figures

For each figure, we check if source data exists and regenerate figures at high resolution (dpi=300).

In [ ]:
# Generate figures
print('Generating figures...')
figures_results = generate_all_figures(str(project_root))

print(f'\nFigure Generation Results:')
print(f'  Total: {len(figures_results)}')
print(f'  Successful: {sum(1 for f in figures_results if f["success"])}')
print(f'  Failed: {sum(1 for f in figures_results if not f["success"])}')

for fig in figures_results:
    status = '✓' if fig['success'] else '✗'
    print(f"\n{status} {fig['name']}")
    if not fig['success']:
        print(f"  Error: {fig['error_msg']}")
    else:
        print(f"  Output: {fig['output_path']}")

## Generate All Tables

Tables are exported as both CSV and LaTeX formats.

In [ ]:
# Generate tables
print('Generating tables...')
tables_results = generate_all_tables(str(project_root))

print(f'\nTable Generation Results:')
print(f'  Total: {len(tables_results)}')
print(f'  Successful: {sum(1 for t in tables_results if t["success"])}')
print(f'  Failed: {sum(1 for t in tables_results if not t["success"])}')

for tbl in tables_results:
    status = '✓' if tbl['success'] else '✗'
    print(f"\n{status} {tbl['name']}")
    if tbl['success']:
        print(f"  CSV: {tbl['output_path_csv']}")
        print(f"  LaTeX: {tbl['output_path_tex']}")
    else:
        print(f"  Error: {tbl['error_msg']}")

## Create Manuscript Manifest

Generate a manifest JSON file summarizing all generated figures and tables.

In [ ]:
# Create manifest
print('Creating manuscript manifest...')
manifest = create_manuscript_manifest(
    figures_results,
    tables_results,
    output_dir=str(project_root / 'outputs/reporting')
)

print(f'\nManuscript Manifest Summary:')
print(f'  Created: {manifest["created_at"]}')
print(f'  Analysis: {manifest["analysis"]}')
print(f'\n  Figures:')
print(f'    Generated: {manifest["summary"]["n_figures_generated"]}')
print(f'    Failed: {manifest["summary"]["n_figures_failed"]}')
print(f'\n  Tables:')
print(f'    Generated: {manifest["summary"]["n_tables_generated"]}')
print(f'    Failed: {manifest["summary"]["n_tables_failed"]}')

manifest_path = project_root / 'outputs/reporting/manuscript_manifest.json'
print(f'\nManifest written to: {manifest_path}')

## Summary

All figures and tables have been generated and organized for manuscript submission.

In [ ]:
# Display final summary
print('\n=== MANUSCRIPT GENERATION COMPLETE ===')
print(f'\nFigures Directory: {project_root / "nature_methods/figures"}')
print(f'Tables Directory: {project_root / "nature_methods/tables"}')
print(f'Manifest: {project_root / "outputs/reporting/manuscript_manifest.json"}')

# Count output files
figures_dir = project_root / 'nature_methods/figures'
tables_dir = project_root / 'nature_methods/tables'

if figures_dir.exists():
    n_figs = len(list(figures_dir.glob('*.png')))
    print(f'\nGenerated {n_figs} figures')

if tables_dir.exists():
    n_csv = len(list(tables_dir.glob('*.csv')))
    n_tex = len(list(tables_dir.glob('*.tex')))
    print(f'Generated {n_csv} CSV tables and {n_tex} LaTeX tables')